In [ ]:
import pandas as pd

anio = 2025
mes = "Diciembre"
trm = 3791.47
#archivo = "C:\\Users\\User\\Downloads\\CFO DICIEMBRE 2025.xlsx"

# 1. Leer datos desde Excel
#df = pd.read_excel(archivo, sheet_name="BASE DICIEMBRE ", header=None)
#df.head()
df = df.copy()

In [ ]:
# --- Quitar las 4 primeras filas ---
df = df.iloc[4:].reset_index(drop=True)

In [ ]:
# --- Promover encabezados ---
df.columns = df.iloc[0]      # primera fila como encabezados
df = df.drop(df.index[0]).reset_index(drop=True)
df.head()

In [ ]:
# Conversión robusta de columnas numéricas
num_cols_int = ["Código cuenta contable", "id_EEFF", "1 Nivel", "2 Nivel", "4 Nivel"]
num_cols_float = ["Saldo inicial", "Movimiento débito", "Movimiento crédito", "Saldo final", "MOV Mes"]

for col in num_cols_int:
    df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")

for col in num_cols_float:
    df[col] = pd.to_numeric(df[col], errors="coerce").astype(float)

# Las demás columnas quedan como string
str_cols = ["Nivel", "Transaccional", "Nombre cuenta contable", "Identificación", 
            "Nombre tercero", "NombreCuenta ", "CuentaFin", "SubCuentaFin", 
            "Tipo Costo", "Region ", "Empresa"]

df[str_cols] = df[str_cols].astype("string")


In [ ]:
# --- Reordenar columnas ---
df = df[[
    "id_EEFF", "NombreCuenta ", "CuentaFin", "SubCuentaFin", "Tipo Costo", 
    "Nombre tercero", "Código cuenta contable", "Nombre cuenta contable", 
    "Saldo inicial", "Movimiento débito", "Movimiento crédito", "Saldo final", "Empresa"
]]

In [ ]:
# --- Reemplazar valores en Empresa ---
df["Empresa"] = df["Empresa"].replace({
    "HOLOVERSITY SAS": "HOLOVERSITY",
    "ALIANZA SUPERIOR SAS": "ALIANZA",
    "EDTECH4LIFE SL": "EDTECH4LIFE",
    "VIRTUOSITY 11.11 LLC": "VIRTUOSITY"
})
df.head()

In [ ]:
# --- Eliminar filas en blanco ---
df = df.dropna(how="all")  

In [ ]:
# --- Poner en mayúsculas cada palabra en "Nombre tercero" ---
df["Nombre tercero"] = df["Nombre tercero"].str.title()

In [ ]:
# --- Cambiar tipos adicionales ---
df = df.astype({
    "Código cuenta contable": "string",
    "Saldo inicial": "float",
    "Movimiento débito": "float",
    "Movimiento crédito": "float",
    "Saldo final": "float"
})

In [ ]:
# --- Agregar columna Largo ---
df["Largo"] = df["Código cuenta contable"].str.len()

In [ ]:
# --- Reordenar columnas ---
df = df[[
    "Largo", "Empresa", "id_EEFF", "NombreCuenta ", "CuentaFin", "SubCuentaFin", 
    "Tipo Costo", "Nombre tercero", "Código cuenta contable", "Nombre cuenta contable", 
    "Saldo inicial", "Movimiento débito", "Movimiento crédito", "Saldo final"
]]

In [ ]:
# --- Agregar columna Cuenta según primer dígito ---
df["Cuenta"] = df["Código cuenta contable"].str[0].map({
    "1": "Activo", "2": "Pasivo", "3": "Patrimonio", 
    "4": "Ingreso", "5": "Gasto"
}).fillna("Costo")

In [ ]:
# --- Agregar Estado financiero ---
df["Estado financiero"] = df["Cuenta"].apply(
    lambda x: "Balance general" if x in ["Activo", "Pasivo", "Patrimonio"] else "Estado de resultados"
)

In [ ]:
# --- Agregar Región ---
df["Región"] = df["Empresa"].apply(lambda x: "EUROPA" if x == "EDTECH4LIFE" else "LATAM")

In [ ]:
# --- Agregar Dato financiero ---
def calc_dato(row):
    if row["Cuenta"] == "Activo":
        return row["Saldo final"]
    elif row["Cuenta"] in ["Pasivo", "Patrimonio"]:
        return row["Saldo final"] * -1
    elif row["Cuenta"] == "Ingreso":
        return row["Movimiento crédito"] - row["Movimiento débito"]
    else:  # Gasto o Costo
        return row["Movimiento débito"] - row["Movimiento crédito"]
df["Dato financiero"] = df.apply(calc_dato, axis=1)


In [ ]:
# --- Reordenar columnas finales ---
df = df[[
    "Largo", "Estado financiero", "Empresa", "Región", "Cuenta", "id_EEFF", "NombreCuenta ",
    "CuentaFin", "SubCuentaFin", "Tipo Costo", "Nombre tercero", "Código cuenta contable",
    "Nombre cuenta contable", "Dato financiero"
]]

In [ ]:
# Insertar Año y Mes
pos = df.columns.get_loc("Región") + 1
df.insert(pos, "Año", anio)
df.insert(pos, "Mes", mes)

In [ ]:
# Añadir columna TRM con valor fijo
df["TRM"] = trm

# Añadir columna Dato Financiero USD (Dato financiero / TRM)
df["Dato Financiero USD"] = df["Dato financiero"] / df["TRM"]

In [ ]:
df.head()

In [ ]:
# Exportar a Excel
df.to_excel(
    r"C:\Users\User\Downloads\limpiado_Griky.xlsx",  # Ruta de salida
    index=False                                      # No guardar índice
)
df.head()